# Tutorial 2: Hooks and Caching — Looking Inside the Model

**Prerequisites:** T01 (Transformers from Scratch)

**What you'll learn:**
- How IRTK's hook system lets you capture any intermediate activation
- How to modify activations mid-computation (interventions)
- The difference between caching (observing) and hooking (intervening)
- Practical patterns you'll use in every investigation

---

## Why Hooks?

A transformer is a pipeline of computations. To understand what it's doing, we need to:
1. **Observe**: Capture intermediate activations at any point
2. **Intervene**: Modify activations to test causal hypotheses

IRTK provides both through its **hook system**. Every named computation in the model has a hook point where you can capture or modify the activation.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.hook_points import HookState

cfg = HookedTransformerConfig(
    n_layers=2, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 42, 17, 88, 55])
print('Model ready')

## Method 1: `run_with_cache` — Capture Everything

The simplest approach: run the model and get back every intermediate activation.

In [ ]:
logits, cache = model.run_with_cache(tokens)

print(f'Captured {len(cache)} activations:')
print()
for key_name in sorted(cache.keys()):
    shape = cache[key_name].shape
    print(f'  {key_name}: {shape}')

### Understanding Hook Point Names

Hook names follow a consistent pattern:

| Pattern | What it captures | Shape |
|---------|-----------------|-------|
| `blocks.{L}.hook_resid_pre` | Residual before layer L | `[seq, d_model]` |
| `blocks.{L}.hook_resid_mid` | Residual after attention, before MLP | `[seq, d_model]` |
| `blocks.{L}.hook_resid_post` | Residual after full layer L | `[seq, d_model]` |
| `blocks.{L}.hook_attn_out` | Attention output (before adding to residual) | `[seq, d_model]` |
| `blocks.{L}.hook_mlp_out` | MLP output (before adding to residual) | `[seq, d_model]` |
| `blocks.{L}.attn.hook_q` | Query vectors | `[seq, n_heads, d_head]` |
| `blocks.{L}.attn.hook_k` | Key vectors | `[seq, n_heads, d_head]` |
| `blocks.{L}.attn.hook_v` | Value vectors | `[seq, n_heads, d_head]` |
| `blocks.{L}.attn.hook_pattern` | Attention patterns | `[n_heads, seq, seq]` |
| `blocks.{L}.attn.hook_z` | Head outputs (pre-projection) | `[seq, n_heads, d_head]` |
| `blocks.{L}.mlp.hook_pre` | MLP pre-activation | `[seq, d_mlp]` |
| `blocks.{L}.mlp.hook_post` | MLP post-activation | `[seq, d_mlp]` |

In [ ]:
# Example: Get the attention pattern for layer 0
pattern = cache['blocks.0.attn.hook_pattern']
print(f'Attention pattern shape: {pattern.shape}')
print(f'  [{cfg.n_heads} heads, {len(tokens)} query positions, {len(tokens)} key positions]')
print()

# Example: Get the residual stream after layer 1
resid = cache['blocks.1.hook_resid_post']
print(f'Final residual shape: {resid.shape}')
print(f'  [{len(tokens)} positions, {cfg.d_model} dimensions]')

## Method 2: `run_with_hooks` — Intervene on Activations

This is the power tool. You provide **hook functions** that can modify activations as the model runs. This lets you test causal hypotheses:

- "If I zero out this attention head, does the model still get the answer right?"
- "If I replace this layer's output with a different run's output, what changes?"

A hook function takes `(activation, hook_name)` and returns the modified activation (or `None` to leave it unchanged).

In [ ]:
# Example: Zero out all of layer 0's attention output
def zero_attention(x, name):
    """Replace the attention output with zeros."""
    return jnp.zeros_like(x)

# Run with the hook
hooks = {'blocks.0.hook_attn_out': zero_attention}
logits_modified = model.run_with_hooks(tokens, fwd_hooks=hooks)

# Compare
logits_normal = model(tokens)
diff = float(jnp.linalg.norm(logits_modified[-1] - logits_normal[-1]))
print(f'Logit difference from zeroing layer 0 attention: {diff:.4f}')
print('A large difference means layer 0 attention matters for the prediction!')

In [ ]:
# Example: Add a steering vector to the residual stream
def add_direction(x, name):
    """Add a direction vector to the residual stream."""
    direction = jnp.array([1.0 if i % 2 == 0 else -1.0 for i in range(cfg.d_model)])
    return x + 0.5 * direction  # Add to all positions

hooks = {'blocks.0.hook_resid_mid': add_direction}
logits_steered = model.run_with_hooks(tokens, fwd_hooks=hooks)

# How much did the prediction change?
diff = float(jnp.linalg.norm(logits_steered[-1] - logits_normal[-1]))
print(f'Logit difference from steering: {diff:.4f}')

## Method 3: Low-Level `HookState` (For Advanced Use)

Under the hood, `run_with_cache` and `run_with_hooks` use IRTK's `HookState` object. You can use it directly for maximum control.

In [ ]:
# Capture only specific activations (more efficient for large models)
my_cache = {}
hook_state = HookState(hook_fns={}, cache=my_cache)
logits = model(tokens, hook_state=hook_state)

# my_cache now has everything (same as run_with_cache)
print(f'Captured {len(my_cache)} activations')

# You can also combine caching WITH hooks
my_cache2 = {}

def spy_and_modify(x, name):
    """Log the norm AND modify the activation."""
    print(f'  Hook fired: {name}, norm={float(jnp.linalg.norm(x)):.4f}')
    return x * 1.1  # Scale up by 10%

hook_state2 = HookState(
    hook_fns={'blocks.0.hook_attn_out': spy_and_modify},
    cache=my_cache2,
)
print('\nRunning with spy hook:')
logits = model(tokens, hook_state=hook_state2)

## Practical Patterns

Here are the patterns you'll use over and over in mechinterp research:

In [ ]:
# Pattern 1: Ablation ("does this component matter?")
# Zero out a specific component and measure the effect

logits_base = model(tokens)

for layer in range(cfg.n_layers):
    for component in ['hook_attn_out', 'hook_mlp_out']:
        hook_name = f'blocks.{layer}.{component}'
        hooks = {hook_name: lambda x, n: jnp.zeros_like(x)}
        logits_abl = model.run_with_hooks(tokens, fwd_hooks=hooks)
        diff = float(jnp.linalg.norm(logits_abl[-1] - logits_base[-1]))
        name = 'Attn' if 'attn' in component else 'MLP'
        print(f'  Layer {layer} {name} ablation effect: {diff:.4f}')

In [ ]:
# Pattern 2: Activation patching ("is this activation carrying the signal?")
# Run on two inputs, swap one activation from run A into run B

tokens_a = jnp.array([1, 42, 17, 88, 55])
tokens_b = jnp.array([50, 60, 70, 80, 90])

_, cache_a = model.run_with_cache(tokens_a)
logits_b = model(tokens_b)

# Patch layer 0 attention output from run A into run B
patched_act = cache_a['blocks.0.hook_attn_out']
hooks = {'blocks.0.hook_attn_out': lambda x, n: patched_act}
logits_patched = model.run_with_hooks(tokens_b, fwd_hooks=hooks)

diff = float(jnp.linalg.norm(logits_patched[-1] - logits_b[-1]))
print(f'Effect of patching layer 0 attn from run A into run B: {diff:.4f}')
print('Large effect = this activation carries important information.')

## Key Takeaways

1. **`run_with_cache(tokens)`** — captures all activations for analysis
2. **`run_with_hooks(tokens, fwd_hooks={...})`** — modifies activations for causal testing
3. **Hook names** follow `blocks.{layer}.{component}` convention
4. **Hook functions** take `(activation, name)` and return modified activation or `None`
5. **Ablation** (zeroing) tells you if a component matters
6. **Patching** (swapping) tells you what information a component carries

**Next: [T03 — The Residual Stream](T03_the_residual_stream.ipynb)** — Deep dive into the central object of mechanistic interpretability.